cuMFFIMiner

A short tutorial for cuMFFIMiner. See the [algorithm guide](https://github.com/udayRage/cuPAMI/blob/main/docs/algorithms/cuMFFIMiner.md) for details.

In Colab, choose Runtime → Change runtime type → GPU. Install the library from GitHub and a CuPy wheel matching your CUDA installation. The setup below uses CUDA 12. Run setup once; skip it if your local environment is already configured.

In [1]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("cupami") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "git+https://github.com/udayRage/cuPAMI.git"])
if importlib.util.find_spec("cupy") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "cupy-cuda12x"])

In [2]:
import cupy as cp

print("CUDA devices:", cp.cuda.runtime.getDeviceCount())


CUDA devices: 2


Prepare fuzzy input

There is no dedicated fuzzy data generator. The small sample below is ready to run. Replace it with your transformed data when using your own dataset. Each row contains labels, a colon, and matching memberships, for example `1.L\t2.H:0.8\t0.6`. Memberships must be between 0 and 1. `1` is the base item and `L` is its fuzzy region. The sample contains three transactions.

In [3]:
from pathlib import Path
from cupami import cuMFFIMiner

fuzzy_rows = [
    "1.L\t1.H\t2.H:0.8\t0.2\t0.6",
    "1.L\t1.H\t2.H:0.7\t0.3\t0.9",
    "1.L\t1.H\t2.H:0.4\t0.6\t0.8",
]
if not fuzzy_rows:
    raise ValueError("Add your fuzzy transaction rows above before continuing.")

input_file = Path("fuzzy.csv")
input_file.write_text("\n".join(fuzzy_rows) + "\n", encoding="utf-8")


72

Mine patterns

`minSup=0.1` means 10% of the transaction count as fuzzy support. Support sums the minimum membership of the pattern in each transaction.

In [4]:
miner = cuMFFIMiner(str(input_file), minSup=0.1)
miner.mine()


View results

Show the first ten patterns and their supports.

In [5]:
from itertools import islice

patterns = miner.getPatterns()
print("Patterns:", len(patterns))
for pattern, support in islice(patterns.items(), 10):
    print(pattern, support)


Patterns: 5
1.H 1.1
1.L 1.9
2.H 2.3
1.H	2.H 1.1
1.L	2.H 1.7


Timing and output

Times are in seconds. Save the patterns to a local text file.

In [6]:
print("Loading:", miner.getLoadingTime())
print("Mining:", miner.getMiningTime())
print("Total:", miner.getTotalRuntime())
miner.save("patterns.txt")


Loading: 0.00027733994647860527
Mining: 0.24120780592784286
Total: 0.24148514587432146
